In [2]:
%reload_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from my_graphrag import KBGraphRAG

load_dotenv(".env", override=True)

True

In [3]:
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()

kb = KBGraphRAG(driver=driver, max_workers=10)
print(kb.model)

openai/gpt-oss-120b


In [6]:
#Define example texts and entity types
example_texts = [
    "Tom is an American",
    "Tom lives in Washington DC", 
    "Tom went to school in Utah"
],
allowed_entities = ["Person", "Nationality", "Location"]
#Extract entities and relationships
result = kb.extract_nodes_and_rels(example_texts, allowed_entities)
print(result)
kb.query("MATCH (e:__Entity__) RETURN e.name, e.description LIMIT 20")

<coroutine object KBGraphRAG.extract_nodes_and_rels at 0x110949000>


[{'e.name': 'TOM',
  'e.description': ['Tom is a person who lives in Washington DC and attended school in Utah']},
 {'e.name': 'WASHINGTON DC',
  'e.description': ['Washington DC is the capital city of the United States where Tom resides']},
 {'e.name': 'UTAH',
  'e.description': ['Utah is a U.S. state where Tom went to school']}]

In [7]:
await kb.summarize_nodes_and_rels()

Summarizing nodes: 0it [00:00, ?it/s]
Summarizing relationships: 0it [00:00, ?it/s]


'Successfuly summarized nodes and relationships'

In [8]:
await kb.summarize_communities()

Leiden algorithm identified 1 community levels with 1 communities on the last level.


Summarizing communities: 100%|██████████| 1/1 [00:02<00:00,  2.99s/it]


'Generated 1 community summaries'

In [18]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"]
)
print("Connected to Neo4j")

Connected to Neo4j


In [9]:
from langchain_core.documents import Document

movie_texts = [
    "Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.",
    "Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.",
    "Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008."
]

documents = [Document(page_content=text) for text in movie_texts]

documents

[Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'),
 Document(metadata={}, page_content='Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'),
 Document(metadata={}, page_content='Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.')]

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)


Chunk 1: Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.
Chunk 2: Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.
Chunk 3: Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.


In [15]:
from langchain_groq import ChatGroq
from langchain_neo4j import LLMGraphTransformer

llm = ChatGroq(model=kb.model)

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)
graph_documents


[GraphDocument(nodes=[Node(id='Christopher Nolan', type='Person', properties={}), Node(id='Leonardo Dicaprio', type='Person', properties={}), Node(id='Inception', type='Movie', properties={}), Node(id='Science Fiction', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Christopher Nolan', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Leonardo Dicaprio', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='Inception', type='Movie', properties={}), target=Node(id='Science Fiction', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.')),
 GraphDocument(nodes=[Node(id='Christopher No

In [16]:
# Inspect extracted graph data
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)


--- Graph Document 1 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Leonardo Dicaprio' type='Person' properties={}
id='Inception' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Leonardo Dicaprio', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='Inception', type='Movie', properties={}) target=Node(id='Science Fiction', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Matthew Mcconaughey' type='Person' properties={}
id='Interstellar' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properti

In [20]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)
print("Graph data added to Neo4j")

Graph data added to Neo4j


In [21]:
graph.refresh_schema()
print(graph.schema)

Node properties:
__Chunk__ {id: STRING, text: STRING}
__Community__ {id: STRING, summary: STRING, level: INTEGER, title: STRING, rating: FLOAT, rating_explanation: STRING}
PERSON {name: STRING, description: LIST, summary: STRING, communities: LIST}
LOCATION {name: STRING, description: LIST, summary: STRING, communities: LIST}
Document {id: STRING, text: STRING}
Person {id: STRING}
Movie {id: STRING}
Genre {id: STRING}
Relationship properties:
RELATIONSHIP {description: STRING, strength: INTEGER}
SUMMARIZED_RELATIONSHIP {summary: STRING}
The relationships:
(:__Chunk__)-[:MENTIONS]->(:PERSON)
(:__Chunk__)-[:MENTIONS]->(:LOCATION)
(:PERSON)-[:RELATIONSHIP]->(:LOCATION)
(:PERSON)-[:SUMMARIZED_RELATIONSHIP]->(:LOCATION)
(:PERSON)-[:IN_COMMUNITY]->(:__Community__)
(:LOCATION)-[:IN_COMMUNITY]->(:__Community__)
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENR

In [22]:
result = graph.query("""
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
OPTIONAL MATCH (actor:Person)-[:ACTED_IN]->(movie)
RETURN director.id AS director, movie.id AS movie, collect(actor.id) AS actors
""")

result

/var/folders/ls/grnwd6y54m58zjpyhg_hg8880000gn/T/ipykernel_7722/3832834637.py:1: RuntimeWarning: coroutine 'KBGraphRAG.extract_nodes_and_rels' was never awaited
  result = graph.query("""


[{'director': 'Christopher Nolan',
  'movie': 'Inception',
  'actors': ['Leonardo Dicaprio']},
 {'director': 'Christopher Nolan',
  'movie': 'Interstellar',
  'actors': ['Matthew Mcconaughey']},
 {'director': 'Christopher Nolan',
  'movie': 'The Dark Knight',
  'actors': ['Christian Bale']}]

In [23]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)
response = chain.invoke({
    "query": "Which movies did Christopher Nolan direct?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
RETURN m.id AS movieId, m.title AS title;
Full Context:
[{'movieId': 'Inception', 'title': None}, {'movieId': 'Interstellar', 'title': None}, {'movieId': 'The Dark Knight', 'title': None}]

> Finished chain.
I don't know the answer.


In [24]:
response = chain.invoke({
    "query": "Which movie did Christopher Nolan direct in 2010 and who acted in it?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie)
WHERE m.id CONTAINS '2010'
MATCH (a:Person)-[:ACTED_IN]->(m)
RETURN m.id AS movieId, collect(a.name) AS actors
Full Context:
[]

> Finished chain.
I don't know the answer.
